## sLTN Tutorial Series

This is the first of six tutorials introducing **sLTN** (Structural Logic Tensor Networks):

1. **Signature & Interpretation** (this notebook): the vocabulary layer
2. [FOL Formulas](2-FOLFormulas.ipynb): connectives and quantifiers
3. [Fuzzy Operators & Gradients](3-FuzzyOperators&Gradients.ipynb): choosing operators for learning
4. [KB & Learning](4-KB&Learning.ipynb): knowledge bases and training loops
5. [Structural LTN](5-StructuralLTN.ipynb): reasoning over time, sequences and graphs
6. [Advanced Features](6-AdvancedFeatures.ipynb): language extensions, advanced structure, training, deployment and debugging

The notebook dependencies are installed with:

```bash
pip install -e ".[notebooks]"
```

# Tutorial 1 — The Vocabulary: Sorts, Symbols, and Interpretation

sLTN separates **syntax** from **semantics**.

- A `Signature` declares the vocabulary of a logical language, that is, the names of the symbols and their sorts.
- An `Interpretation` assigns to each symbol `a` of the signature a *grounding*, written 𝓘[a]. An individual is grounded with a tensor, and a predicate with a neural network or a Python function.

The groundings determine, by structural recursion, the *denotation* 𝓘(e) of every term and formula `e`. A denotation is an annotated tensor, that is, a tensor whose axes have names and roles. Groundings are given by the user, and denotations are computed by sLTN.

A formula is therefore written and checked once against the signature, and its denotation can then be computed under any interpretation, for instance on other data or with another network.

| Symbol | Declares | Typical grounding |
|---|---|---|
| Sort | A domain of individuals | A `Type`, the shape of one individual |
| Constant | One named individual | A tensor, or an `nn.Parameter` if it is learned |
| Variable | A placeholder for individuals of a sort | A batch of individuals (a tensor) |
| Predicate | A map from individuals to a truth degree in [0, 1] | A neural module or a callable |
| Function | A map from individuals to an individual | A neural module or a callable |

The notebook is organised accordingly.

1. **Signature.** Declaration of the vocabulary and parsing of formulas, without any values.
2. **Interpretation.** Grounding of every symbol.
3. **Denotation.** Computation of the denotations of terms and formulas, and the correspondence between their axes and the free variables.

In [ ]:
import torch
import torch.nn as nn
import sltn
from sltn import Signature, Interpretation, Type

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Part 1 — The Signature

### 1.1 Sorts

A **sort** is a domain of individuals, such as animals, people, images or embeddings. Every other symbol is declared with respect to sorts. A constant or a variable belongs to a sort, a predicate takes arguments of given sorts, and a function maps input sorts to an output sort.

The following cell declares two sorts, `Animal` and `Embedding`, the latter for a learned representation of an animal.

In [ ]:
sig = Signature("tutorial")

sig.sort("Animal")
sig.sort("Embedding")

### 1.2 Constants and variables

A **constant** names one fixed individual, such as `Fido`. A **variable** ranges over the individuals of a sort. It is grounded with a finite batch of individuals, and a formula with a free variable is evaluated on each of them.

In [ ]:
sig.constant("Fido", "Animal")
sig.constant("Rex", "Animal")   # its features will be learned

sig.variable("x", "Animal")
sig.variable("y", "Animal")

### 1.3 Predicates and functions

A **predicate** is declared with its name and the sorts of its arguments. A **function** is declared, in addition, with an output sort.

In [ ]:
sig.predicate("IsDog", ["Animal"])
sig.predicate("Near", ["Animal"])               # close to a reference point
sig.predicate("Similar", ["Animal", "Animal"])

sig.function("embed", ["Animal"], "Embedding")
sig.function("diff", ["Animal", "Animal"], "Animal")

print(sig)

`Bool` and `Real` are built-in sorts, present in every signature. `Bool` is the sort of truth degrees, and `Real` the sort of a single real number.

### 1.4 Parsing formulas

The signature suffices to parse a formula. Parsing resolves every name against the signature and builds a formula tree. No symbol has been grounded at this point.

In [ ]:
phi = sig.parse("Similar(x, Fido)")
print(phi, "   ->", type(phi).__name__)

print(sig.parse("IsDog(diff(x, Rex))"))   # terms can be nested

The parser checks the formula against the declarations before any data or network is involved. Every name must be declared.

In [ ]:
try:
    sig.parse("Barks(x)")
except NameError as e:
    print("NameError:", e)

Each predicate and function must receive the declared number of arguments, each of the declared sort. The term `embed(x)` has sort `Embedding`, not `Animal`.

In [ ]:
for formula in ["Similar(x)", "IsDog(embed(x))"]:
    try:
        sig.parse(formula)
    except TypeError as e:
        print("TypeError:", e)

## Part 2 — The Interpretation

An `Interpretation` is built on a signature and assigns the grounding 𝓘[a] of a symbol `a` with `interp["a"] = value`. It is an `nn.Module`, hence it inherits all the functionality of a PyTorch module, such as `parameters()`, `to(device)`, `state_dict()`, `train()` and `eval()`, which act on the registered groundings (2.4).

### 2.1 Sorts

A sort is grounded with a `Type`, which specifies the **shape of one individual** and a name for its feature axis. In this tutorial an animal is a vector in ℝ² of two features, its size and its furriness, and an embedding is a vector in ℝ⁸. The built-in sorts `Bool` and `Real` are grounded with `sltn.Bool` and `sltn.Real`. Every sort has at least one feature axis. A scalar sort therefore uses `shape=(1,)`, as `sltn.Real` and `sltn.Bool` do, and a batch of scalars has shape `(B, 1)`, not `(B,)`.

In [ ]:
interp = Interpretation(sig).to(device)

interp["Animal"] = Type("R2", shape=(2,))
interp["Embedding"] = Type("R8", shape=(8,))

print(sltn.Bool)
print(sltn.Real)

### 2.2 Constants

A constant is grounded with a tensor of **exactly the shape of its sort**, here `(2,)`, without a batch axis. A constant whose features are to be learned is grounded with an `nn.Parameter`.

In [ ]:
interp["Fido"] = torch.tensor([2.1, 3.0])
interp["Rex"] = nn.Parameter(torch.zeros(2))

print([name for name, _ in interp.named_parameters()])

The shape is checked against the `Type` of the sort when the constant is grounded.

In [ ]:
try:
    interp["Fido"] = torch.tensor([2.1, 3.0, 0.5])   # an Animal has 2 features, not 3
except ValueError as e:
    print("ValueError:", e)

### 2.3 Variables

A variable is grounded with a batch of individuals, a tensor of shape `[n, *shape of the sort]`. In the following cell `x` is grounded with 10 animals and `y` with 7.

In [ ]:
interp["x"] = torch.randn(10, 2)
interp["y"] = torch.randn(7, 2)

### 2.4 Predicates

A predicate is grounded with an `nn.Module`, if it is to be learned, or with a plain callable, if it is fixed. The grounding receives one tensor per argument, of shape `[B, *shape of the argument's sort]`, where `B` is the number of inputs evaluated together. It returns a tensor of shape `[B, 1]` containing one truth degree in [0, 1] per input. The final axis of size 1 is required.

In [ ]:
# A trainable unary predicate
class IsDogNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 16), nn.ELU(), nn.Linear(16, 1), nn.Sigmoid())

    def forward(self, a):                  # a: [B, 2]
        return self.net(a)                 # [B, 1]

interp["IsDog"] = IsDogNet()

# A fixed unary predicate: the closer to mu, the higher the truth degree
mu = torch.tensor([0.5, 0.5], device=device)
interp["Near"] = lambda a: torch.exp(-torch.norm(a - mu, dim=-1, keepdim=True))

# A trainable binary predicate: one tensor per argument
class SimilarNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 8), nn.ELU(), nn.Linear(8, 1), nn.Sigmoid())

    def forward(self, a, b):               # a, b: [B, 2]
        return self.net(torch.cat([a, b], dim=-1))

interp["Similar"] = SimilarNet()

The interpretation registers every grounding that is an `nn.Module` or an `nn.Parameter`. Registered groundings are moved by `interp.to(device)`, listed by `interp.parameters()` and saved by `state_dict()`. An object captured inside a lambda is not registered. The tensor `mu` above must therefore be created on `device` explicitly, and the parameters of a network captured inside a lambda are absent from `interp.parameters()`, so an optimiser built from them does not train the network.

The following cell shows the correct form. The grounding is an `nn.Module` that stores the network as an attribute, a fixed tensor with `register_buffer`, and a trainable tensor as an `nn.Parameter`.

In [ ]:
class NearNet(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net                      # submodule: registered, hence trained and saved

    def forward(self, a):                   # a: [B, 2]
        return self.net(a)                  # [B, 1]

class NearFixed(nn.Module):
    def __init__(self, mu, learn_mu=False):
        super().__init__()
        if learn_mu:
            self.mu = nn.Parameter(mu)      # trainable centre
        else:
            self.register_buffer("mu", mu)  # fixed centre: moved by .to() and saved in state_dict()

    def forward(self, a):
        return torch.exp(-torch.norm(a - self.mu, dim=-1, keepdim=True))

good = interp.copy()
good["Near"] = NearNet(nn.Sequential(nn.Linear(2, 1), nn.Sigmoid()))
print("Near parameters:", [n for n, _ in good.named_parameters() if n.startswith("Near")])

good["Near"] = NearFixed(torch.tensor([0.5, 0.5]))
print("mu in state_dict:", any(k.endswith("mu") for k in good.state_dict()))

Alternatively, the network can be the grounding of a separate function symbol. In the following cell the network `nn.Linear(2, 1)` is the grounding of a new function symbol `score`, and the formula applies the predicate to its output, as in `High(score(Fido))`. The network is then registered as the grounding of `score`, while the lambda that grounds `High` contains only `torch.sigmoid`.

In [ ]:
sig2 = sltn.Signature()
sig2.sort("Point")
sig2.sort("Score")
sig2.constant("Fido", "Point")
sig2.function("score", ["Point"], "Score")
sig2.predicate("High", ["Score"])

interp2 = sltn.Interpretation(sig2).to(device)
interp2["Point"] = sltn.Type("Point", shape=(2,))
interp2["Score"] = sltn.Type("Score", shape=(1,))
interp2["Fido"]  = torch.tensor([0.2, 0.9], device=device)
interp2["score"] = nn.Linear(2, 1)                  # registered grounding
interp2["High"]  = lambda s: torch.sigmoid(s)        # captures no object

print("parameters:", [n for n, _ in interp2.named_parameters()])
print(interp2("High(score(Fido))"))

In summary, a grounding that uses a network or a tensor is written as an `nn.Module`. A separate function symbol is useful when the output of the network, here the score, is also needed by other formulas. A lambda is used only for groundings without trainable parameters, and any tensor it uses, such as `mu`, must be created on the correct device by hand.

### 2.5 Functions

A function is grounded in the same way. It returns a tensor of shape `[B, *shape of the output sort]`. In the following cell `embed` returns `[B, 8]` and `diff` returns `[B, 2]`.

In [ ]:
interp["embed"] = nn.Linear(2, 8)
interp["diff"] = lambda a, b: a - b

print(interp)

### 2.6 Common mistakes

The shape of a constant is checked when it is grounded (2.2). A predicate or a function, instead, is grounded with a callable, whose output is known only when it is called. Whether a predicate returns a tensor of shape `[B, 1]` with values in [0, 1] (2.4) is therefore checked at the first call, for instance in `interp("Near(x)")` and not at the assignment.

The following cells show three common mistakes: omitting the final axis of size 1, reducing over the batch axis, and returning values outside [0, 1]. The wrong groundings are assigned to `bad`, a copy of `interp`, so that `interp` keeps its correct groundings for the rest of the tutorial.

**Missing final axis of size 1.** The norm below omits `keepdim=True` and drops the last axis, so the predicate returns `[B]` instead of `[B, 1]`.

In [ ]:
bad = interp.copy()
bad["Near"] = lambda a: torch.exp(-torch.norm(a - mu, dim=-1))   # keepdim=True is missing

try:
    bad("Near(x)")
except ValueError as e:
    print("ValueError:", e)

The message reports the predicate, the shape it returned and the expected shape.

**Reduction over the batch axis.** The first axis of each input is the batch axis `B`, not a feature axis. A reduction without `dim`, such as `torch.norm(t)`, `torch.all(t)` or `t.sum()`, also reduces the batch axis and returns one number for the whole batch instead of one per input.

In [ ]:
bad["Similar"] = lambda a, b: torch.exp(-torch.norm(a - b))   # reduces the batch axis as well

try:
    bad("Similar(x, y)")
except ValueError as e:
    print("ValueError:", e)

The grounding received `a` and `b` of shape `[70, 2]`, one row for each of the 10 × 7 pairs of `x` and `y`. It must return one truth degree per row, so the reduction must run over the feature axis only.

In [ ]:
bad["Similar"] = lambda a, b: torch.exp(-torch.norm(a - b, dim=-1, keepdim=True))   # [70, 2] -> [70, 1]

print(tuple(bad("Similar(x, y)").value.shape))

sLTN reshapes the 70 rows into the axes `x` and `y`, which gives `(10, 7, 1)`. Every reduction inside a grounding (`sum`, `mean`, `max`, `all`, ...) must run over the axes of one individual, `dim=-1` here or `dim=(1, 2)` for a sort of shape `(3, 3)`, and never over the first axis.

**Values outside [0, 1].** A truth degree lies in [0, 1]. A network without a final sigmoid usually violates this condition.

In [ ]:
bad["Near"] = lambda a: torch.norm(a - mu, dim=-1, keepdim=True)   # a distance, not a truth degree

try:
    bad("Near(x)")
except ValueError as e:
    print("ValueError:", e)

## Part 3 — Denotations

### 3.1 A denotation is a tensor with named axes

Calling the interpretation on a term or a formula `e`, parsed or given as a string, computes its denotation 𝓘(e). The denotation is an `sltn.Tensor`, a PyTorch tensor (`.value`) whose axes have **names** and **roles**.

- A **variable** axis for each free variable of the formula, with one entry per individual.
- **Domain** axes for the features of one individual. For a formula, this is the `bool` axis of size 1 that holds the truth degree.

A third role, **structural**, is used for time steps or positions in a sequence and is introduced in Tutorial 5.

In [ ]:
result = interp("IsDog(x)")
print(result)
print(result.value.shape)

The variable `x` is grounded as `[10, 2]`, with a variable axis `x` of size 10 and a domain axis `r2` of size 2. `IsDog` replaces the 2 features of each animal with one truth degree, so the denotation of `IsDog(x)` has axes `[x: 10, bool: 1]`, the degree to which each of the 10 animals is a dog.

### 3.2 Several variables

With two free variables, the denotation has one axis per variable, and the grounding of the predicate is applied to **every pair**. `Similar(x, y)` thus calls `SimilarNet` on 10 × 7 = 70 pairs. A constant has no variable axis and adds none.

In [ ]:
for formula in ["Similar(x, y)", "Similar(x, Fido)", "Similar(Fido, Rex)"]:
    r = interp(formula)
    print(f"{formula:20s} axes {[a.name for a in r.axes]}   shape {tuple(r.value.shape)}")

This is the semantics of LTN. The denotation of a formula with free variables contains one truth degree for every combination of their individuals, and `B` in 2.4 is the number of combinations. The cost therefore grows with the **product** of the batch sizes.

### 3.3 Terms

Terms have denotations as well. The denotation of a constant has only its domain axis, the denotation of a function application has the domain axis of the output sort (`r8` for `embed`), and terms can be nested inside predicates.

In [ ]:
print(interp("Fido"))
print(interp("embed(x)"))
print(interp("Near(diff(x, Rex))"))

The two notations must not be confused. `interp[name]` returns the grounding 𝓘[a] of a symbol, the object that was assigned to it, such as a tensor, a module or a callable. `interp(e)` returns the denotation 𝓘(e) of an expression, an `sltn.Tensor`. For a variable, 𝓘[x] is a plain tensor of shape `[n, 2]`, whereas 𝓘(x) is the same tensor with a variable axis `x` and a domain axis `r2`.

In [ ]:
print(type(interp["x"]), tuple(interp["x"].shape))
print(interp("x"))

### 3.4 The same formula on other data

A formula is independent of the data. Grounding the variable with another batch and computing the denotation again is sufficient.

In [ ]:
phi = sig.parse("Near(x)")
print(interp(phi).value.shape)

interp["x"] = torch.randn(5, 2)    # a batch of 5 animals
print(interp(phi).value.shape)

### 3.5 The same formula with other groundings

A formula is also independent of the groundings. `copy()` returns an interpretation that shares the groundings of the original, and replacing a grounding in the copy leaves the original unchanged. `interp.with_updates(Near=...)` copies the interpretation and replaces the grounding of `Near` in one call.

In [ ]:
wide = interp.copy()
wide["Near"] = lambda a: torch.exp(-0.1 * torch.norm(a - mu, dim=-1, keepdim=True))   # a wider notion of "near"

print("interp:", interp(phi).value.flatten())
print("wide  :", wide(phi).value.flatten())

## Part 4 — Reading groundings and devices

`interp[name]` returns the grounding of a symbol, for instance to inspect a trained network, a learned constant or a batch.

In [ ]:
print("IsDog :", interp["IsDog"])
print("Rex   :", interp["Rex"])                       # nn.Parameter
print("x     :", tuple(interp["x"].shape), interp["x"].device)

print("devices:", interp.device, next(interp["IsDog"].parameters()).device, next(interp["embed"].parameters()).device)

Only groundings are stored and learned, whereas denotations are recomputed at every call.

## Summary

- A `Signature` declares sorts, constants, variables, predicates and functions. Formulas are parsed and checked against it, without any values.
- An `Interpretation` assigns a grounding 𝓘[a] to every symbol. Sorts are grounded with a `Type`, individuals with tensors of the shape of their sort, and predicates and functions with modules or callables.
- The grounding of a predicate receives `[B, *shape of the sort]` per argument and returns `[B, 1]`, with values in [0, 1].
- The groundings determine the denotation 𝓘(e) of every term and formula, an `sltn.Tensor` with one variable axis per free variable. `interp[name]` returns a grounding and `interp(e)` a denotation.
- The denotation of the same formula can be computed on other data or with other groundings without modifying the formula.

---

**Next:** [Tutorial 2 — FOL Formulas](2-FOLFormulas.ipynb) combines predicates into formulas with connectives and quantifiers.